In [3]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path.cwd().parent

cases = pd.read_csv(
    ROOT / "data/synthetic/disease_cases.csv",
    parse_dates=["year_month"]
)

climate = pd.read_csv(
    ROOT / "data/synthetic/climate_history.csv",
    parse_dates=["year_month"]
)

# Use baseline data only for initial model development.
# Climate-altered scenarios will be evaluated separately.

baseline_cases = cases[
    cases["scenario_id"] == "baseline"
].copy()

baseline_climate = climate[
    climate["scenario_id"] == "baseline"
].copy()

model_data = baseline_cases.merge(
    baseline_climate[
        [
            "year_month",
            "district_id",
            "rainfall_mm",
            "mean_temperature_c",
            "relative_humidity_pct"
        ]
    ],
    on=["year_month", "district_id"],
    how="left",
    validate="many_to_one"
)

model_data = model_data.sort_values(
    ["disease_id", "district_id", "year_month"]
).reset_index(drop=True)

print("=== CLIMATE-AWARE MODELLING DATA ===")
print("Rows:", len(model_data))
print("Columns:", len(model_data.columns))
print("Diseases:", model_data["disease_id"].unique())
print("Districts:", model_data["district_id"].nunique())

print("\nMissing climate values:")
print(
    model_data[
        [
            "rainfall_mm",
            "mean_temperature_c",
            "relative_humidity_pct"
        ]
    ].isna().sum()
)

display(model_data.head())


=== CLIMATE-AWARE MODELLING DATA ===
Rows: 1200
Columns: 10
Diseases: ['dengue' 'malaria']
Districts: 5

Missing climate values:
rainfall_mm              0
mean_temperature_c       0
relative_humidity_pct    0
dtype: int64


,year_month,district_id,disease_id,scenario_id,cases,cases_per_100k,data_type,rainfall_mm,mean_temperature_c,relative_humidity_pct
0,2015-01-01,cbe,dengue,baseline,120,3.429,synthetic_observed,12.3,25.2,61.0
1,2015-02-01,cbe,dengue,baseline,106,3.029,synthetic_observed,19.9,26.4,63.4
2,2015-03-01,cbe,dengue,baseline,68,1.943,synthetic_observed,32.7,27.5,63.7
3,2015-04-01,cbe,dengue,baseline,67,1.914,synthetic_observed,70.6,27.6,65.3
4,2015-05-01,cbe,dengue,baseline,62,1.771,synthetic_observed,70.7,30.1,67.2


In [4]:

# Create climate-lag and seasonal features

model_features = model_data.copy()

climate_variables = [
    "rainfall_mm",
    "mean_temperature_c",
    "relative_humidity_pct"
]

model_features = model_features.sort_values(
    ["district_id", "disease_id", "year_month"]
).reset_index(drop=True)

# Generate climate lags separately for each district and disease
for variable in climate_variables:
    for lag in range(4):
        model_features[f"{variable}_lag_{lag}"] = (
            model_features.groupby(
                ["district_id", "disease_id"]
            )[variable].shift(lag)
        )

# Represent annual seasonality using cyclical features
model_features["calendar_month"] = (
    model_features["year_month"].dt.month
)

model_features["month_sin"] = np.sin(
    2 * np.pi * model_features["calendar_month"] / 12
)

model_features["month_cos"] = np.cos(
    2 * np.pi * model_features["calendar_month"] / 12
)

lag_columns = [
    f"{variable}_lag_{lag}"
    for variable in climate_variables
    for lag in range(4)
]

print("=== FEATURE ENGINEERING RESULTS ===")
print("Total rows:", len(model_features))
print("Climate lag features:", len(lag_columns))
print("Seasonal features: 2")

print("\nMissing values in lag features:")
print(model_features[lag_columns].isna().sum())

print("\nSample engineered features:")
display(
    model_features[
        [
            "year_month",
            "district_id",
            "disease_id",
            "rainfall_mm_lag_0",
            "rainfall_mm_lag_1",
            "rainfall_mm_lag_2",
            "rainfall_mm_lag_3",
            "month_sin",
            "month_cos"
        ]
    ].head(6)
)


=== FEATURE ENGINEERING RESULTS ===
Total rows: 1200
Climate lag features: 12
Seasonal features: 2

Missing values in lag features:
rainfall_mm_lag_0               0
rainfall_mm_lag_1              10
rainfall_mm_lag_2              20
rainfall_mm_lag_3              30
mean_temperature_c_lag_0        0
mean_temperature_c_lag_1       10
mean_temperature_c_lag_2       20
mean_temperature_c_lag_3       30
relative_humidity_pct_lag_0     0
relative_humidity_pct_lag_1    10
relative_humidity_pct_lag_2    20
relative_humidity_pct_lag_3    30
dtype: int64

Sample engineered features:


,year_month,district_id,disease_id,rainfall_mm_lag_0,rainfall_mm_lag_1,rainfall_mm_lag_2,rainfall_mm_lag_3,month_sin,month_cos
0,2015-01-01,cbe,dengue,12.3,NaN,NaN,NaN,5.000000e-01,8.660254e-01
1,2015-02-01,cbe,dengue,19.9,12.3,NaN,NaN,8.660254e-01,5.000000e-01
2,2015-03-01,cbe,dengue,32.7,19.9,12.3,NaN,1.000000e+00,6.123234e-17
3,2015-04-01,cbe,dengue,70.6,32.7,19.9,12.3,8.660254e-01,-5.000000e-01
4,2015-05-01,cbe,dengue,70.7,70.6,32.7,19.9,5.000000e-01,-8.660254e-01
5,2015-06-01,cbe,dengue,80.5,70.7,70.6,32.7,1.224647e-16,-1.000000e+00


In [5]:

#  Prepare climate-aware model training and testing data

feature_columns = lag_columns + ["month_sin", "month_cos"]

# Remove initial months without complete climate lag history
model_ready = model_features.dropna(
    subset=feature_columns + ["cases_per_100k"]
).copy()

train_model = model_ready[
    model_ready["year_month"] < "2022-01-01"
].copy()

test_model = model_ready[
    model_ready["year_month"] >= "2022-01-01"
].copy()

print("=== CLIMATE-AWARE TRAIN/TEST SPLIT ===")
print("Total usable rows:", len(model_ready))
print("Training rows:", len(train_model))
print("Testing rows:", len(test_model))

print("\n=== ROWS BY DISEASE ===")
print("Training:")
print(train_model["disease_id"].value_counts())

print("\nTesting:")
print(test_model["disease_id"].value_counts())

print("\n=== FEATURE CHECKS ===")
print("Number of model features:", len(feature_columns))
print(
    "Missing feature values:",
    train_model[feature_columns].isna().sum().sum()
    + test_model[feature_columns].isna().sum().sum()
)

print("\nTraining date range:")
print(
    train_model["year_month"].min(),
    "to",
    train_model["year_month"].max()
)

print("\nTesting date range:")
print(
    test_model["year_month"].min(),
    "to",
    test_model["year_month"].max()
)

assert train_model["year_month"].max() < test_model["year_month"].min()
assert not train_model[feature_columns].isna().any().any()
assert not test_model[feature_columns].isna().any().any()


=== CLIMATE-AWARE TRAIN/TEST SPLIT ===
Total usable rows: 1170
Training rows: 810
Testing rows: 360

=== ROWS BY DISEASE ===
Training:
disease_id
dengue     405
malaria    405
Name: count, dtype: int64

Testing:
disease_id
dengue     180
malaria    180
Name: count, dtype: int64

=== FEATURE CHECKS ===
Number of model features: 14
Missing feature values: 0

Training date range:
2015-04-01 00:00:00 to 2021-12-01 00:00:00

Testing date range:
2022-01-01 00:00:00 to 2024-12-01 00:00:00


In [6]:

# Train the first climate-aware model (Dengue)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

disease = "dengue"

dengue_train = train_model[
    train_model["disease_id"] == disease
].copy()

dengue_test = test_model[
    test_model["disease_id"] == disease
].copy()

X_train = dengue_train[feature_columns]
y_train = dengue_train["cases_per_100k"]

X_test = dengue_test[feature_columns]
y_test = dengue_test["cases_per_100k"]

dengue_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

dengue_model.fit(X_train, y_train)

dengue_predictions = dengue_model.predict(X_test)

dengue_mae = mean_absolute_error(y_test, dengue_predictions)
dengue_rmse = np.sqrt(
    mean_squared_error(y_test, dengue_predictions)
)

print("=== DENGUE CLIMATE-AWARE MODEL ===")
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Features:", len(feature_columns))
print("MAE:", round(dengue_mae, 3))
print("RMSE:", round(dengue_rmse, 3))

print("\nFirst 5 predictions:")
display(
    dengue_test[
        ["year_month", "district_id", "cases_per_100k"]
    ].assign(
        predicted_cases_per_100k=dengue_predictions
    ).head()
)


=== DENGUE CLIMATE-AWARE MODEL ===
Training samples: 405
Testing samples: 180
Features: 14
MAE: 1.054
RMSE: 2.183

First 5 predictions:


,year_month,district_id,cases_per_100k,predicted_cases_per_100k
84,2022-01-01,cbe,3.200,4.096710
85,2022-02-01,cbe,2.771,2.993228
86,2022-03-01,cbe,2.286,2.399905
87,2022-04-01,cbe,1.514,1.369489
88,2022-05-01,cbe,1.371,1.287752


In [7]:

#  Compare climate-aware vs seasonal-naive (Dengue)

benchmark_path = (
    ROOT / "data/processed/seasonal_naive_predictions.csv"
)

benchmark = pd.read_csv(
    benchmark_path,
    parse_dates=["year_month"]
)

print("Benchmark columns:", benchmark.columns.tolist())
print("Benchmark rows:", len(benchmark))

display(benchmark.head())


Benchmark columns: ['year_month', 'district_id', 'disease_id', 'scenario_id', 'cases', 'cases_per_100k', 'data_type', 'calendar_month', 'predicted_cases_per_100k']
Benchmark rows: 360


,year_month,district_id,disease_id,scenario_id,cases,cases_per_100k,data_type,calendar_month,predicted_cases_per_100k
0,2022-01-01,cbe,dengue,baseline,112,3.200,synthetic_observed,1,3.029
1,2022-02-01,cbe,dengue,baseline,97,2.771,synthetic_observed,2,2.543
2,2022-03-01,cbe,dengue,baseline,80,2.286,synthetic_observed,3,1.543
3,2022-04-01,cbe,dengue,baseline,53,1.514,synthetic_observed,4,0.800
4,2022-05-01,cbe,dengue,baseline,48,1.371,synthetic_observed,5,1.143


In [8]:

#  Compare climate-aware and seasonal-naive models

# Keep only dengue baseline benchmark records
dengue_benchmark = benchmark[
    (benchmark["disease_id"] == "dengue") &
    (benchmark["scenario_id"] == "baseline")
].copy()

# Prepare climate-aware predictions
climate_results = dengue_test[
    ["year_month", "district_id", "disease_id", "cases_per_100k"]
].copy()

climate_results["climate_prediction"] = dengue_predictions

# Merge predictions on matching observations
comparison = climate_results.merge(
    dengue_benchmark[
        ["year_month", "district_id", "disease_id",
         "predicted_cases_per_100k"]
    ],
    on=["year_month", "district_id", "disease_id"],
    how="inner",
    validate="one_to_one"
)

comparison = comparison.rename(
    columns={
        "predicted_cases_per_100k": "seasonal_naive_prediction"
    }
)

assert len(comparison) == len(dengue_test), (
    "Some test observations did not match the benchmark!"
)

actual = comparison["cases_per_100k"]

results = pd.DataFrame([
    {
        "Model": "Seasonal Naive",
        "MAE": mean_absolute_error(
            actual, comparison["seasonal_naive_prediction"]
        ),
        "RMSE": np.sqrt(mean_squared_error(
            actual, comparison["seasonal_naive_prediction"]
        ))
    },
    {
        "Model": "Climate-Aware Random Forest",
        "MAE": mean_absolute_error(
            actual, comparison["climate_prediction"]
        ),
        "RMSE": np.sqrt(mean_squared_error(
            actual, comparison["climate_prediction"]
        ))
    }
])

print("=== DENGUE MODEL COMPARISON ===")
print("Matched observations:", len(comparison))

display(results.round(3))


=== DENGUE MODEL COMPARISON ===
Matched observations: 180


,Model,MAE,RMSE
0,Seasonal Naive,1.101,2.383
1,Climate-Aware Random Forest,1.054,2.183
